# Module 5 - Data Validation

## 1. What is Data Validation?

Data validation is the process of checking whether data meets predefined rules and requirements.

It helps identify incorrect, inconsistent, invalid, or unexpected values before the data is used for analysis or machine learning.

Common validation checks include range, data type, format, category, null values, business rules, and date validation.

In [1]:
import pandas as pd
import numpy as np

# Load the raw dataset
df = pd.read_csv(r"C:\Users\HP\sprint-5-data-cleaning-preprocessing\data\hotel_bookings.csv")

# Display the first five records
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [2]:
print("Dataset Shape:", df.shape)
print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Dataset Shape: (119390, 32)

Data Types:
hotel                                 str
is_canceled                         int64
lead_time                           int64
arrival_date_year                   int64
arrival_date_month                    str
arrival_date_week_number            int64
arrival_date_day_of_month           int64
stays_in_weekend_nights             int64
stays_in_week_nights                int64
adults                              int64
children                          float64
babies                              int64
meal                                  str
country                               str
market_segment                        str
distribution_channel                  str
is_repeated_guest                   int64
previous_cancellations              int64
previous_bookings_not_canceled      int64
reserved_room_type                    str
assigned_room_type                    str
booking_changes                     int64
deposit_type                       

### Observation

The dataset was checked for its structure, data types, and missing values. These checks help identify data quality issues that need to be validated before further analysis.

## 2. Range Validation

Range validation checks whether numerical values fall within an expected or acceptable range.

For the hotel bookings dataset, range validation can be used to check whether values such as `adults`, `children`, `babies`, and `adr` contain valid values.

Examples:
- Number of adults cannot be negative.
- Number of children cannot be negative.
- Number of babies cannot be negative.
- Average Daily Rate (`adr`) cannot be negative.

In [3]:
def validate_non_negative(value):
    return pd.notna(value) and value >= 0


df['adults_range_valid'] = df['adults'].apply(validate_non_negative)
df['children_range_valid'] = df['children'].apply(validate_non_negative)
df['babies_range_valid'] = df['babies'].apply(validate_non_negative)
df['adr_range_valid'] = df['adr'].apply(validate_non_negative)

print("Adults - Invalid values:", (~df['adults_range_valid']).sum())
print("Children - Invalid values:", (~df['children_range_valid']).sum())
print("Babies - Invalid values:", (~df['babies_range_valid']).sum())
print("ADR - Invalid values:", (~df['adr_range_valid']).sum())

Adults - Invalid values: 0
Children - Invalid values: 4
Babies - Invalid values: 0
ADR - Invalid values: 1


### Observation

Range validation was applied to important numerical columns in the hotel bookings dataset. The validation identifies values that are negative or missing where non-negative values are expected.

## 3. Type Validation

Type validation checks whether the values in a column have the expected data type.

Correct data types are important because incorrect types can cause errors during analysis, calculations, and machine learning.

For the hotel bookings dataset, numerical columns should contain numeric values, while categorical columns should contain appropriate categorical or string values.

numeric_columns = ['adults', 'children', 'babies', 'lead_time', 'adr']

print("Type Validation Results:")

for column in numeric_columns:
    print(f"{column}: {pd.api.types.is_numeric_dtype(df[column])}")

In [4]:
numeric_columns = ['adults', 'children', 'babies', 'lead_time', 'adr']

print("Type Validation Results:")

for column in numeric_columns:
    print(f"{column}: {pd.api.types.is_numeric_dtype(df[column])}")

Type Validation Results:
adults: True
children: True
babies: True
lead_time: True
adr: True


### Observation

The selected numerical columns were checked to confirm whether they contain numeric data types. Proper data types help ensure that calculations and further data processing can be performed correctly.

## 4. Format Validation

Format validation checks whether data follows the expected format or pattern.

For the hotel bookings dataset, format validation can be used for fields such as `reservation_status_date` to ensure that the values can be interpreted as valid dates.

In [5]:
date_values = pd.to_datetime(
    df['reservation_status_date'],
    errors='coerce'
)

invalid_date_count = date_values.isna().sum()

print("Invalid date format values:", invalid_date_count)

Invalid date format values: 0


### Observation

The `reservation_status_date` column was checked for valid date formats. Values that could not be converted to a valid date were identified as invalid.

## 5. Category Validation

Category validation checks whether categorical values belong to a predefined set of accepted categories.

For the hotel bookings dataset, columns such as `hotel`, `meal`, `deposit_type`, and `customer_type` should contain valid category values.

In [6]:
valid_hotel_types = ['Resort Hotel', 'City Hotel']

valid_customer_types = [
    'Contract',
    'Group',
    'Transient',
    'Transient-Party'
]

hotel_valid = df['hotel'].isin(valid_hotel_types)
customer_type_valid = df['customer_type'].isin(valid_customer_types)

print("Invalid hotel category values:", (~hotel_valid).sum())
print("Invalid customer type values:", (~customer_type_valid).sum())

Invalid hotel category values: 0
Invalid customer type values: 0


### Observation

Category validation was applied to the `hotel` and `customer_type` columns using predefined accepted values. Any values outside the accepted categories were identified as invalid.

## 6. Null Validation

Null validation checks whether required fields contain missing or null values.

Missing values can cause problems during data analysis and machine learning. Required columns should be checked to ensure that important information is available.

In [7]:
required_columns = [
    'hotel',
    'arrival_date_year',
    'arrival_date_month',
    'adults',
    'reservation_status'
]

print("Null Validation Results:")

for column in required_columns:
    missing_count = df[column].isnull().sum()
    print(f"{column}: {missing_count} missing values")

Null Validation Results:
hotel: 0 missing values
arrival_date_year: 0 missing values
arrival_date_month: 0 missing values
adults: 0 missing values
reservation_status: 0 missing values


### Observation

The required columns were checked for missing values. Columns containing null values were identified for further data cleaning and handling.

## 7. Business Rule Validation

Business rule validation checks whether the data follows rules based on the real-world meaning of the dataset.

For the hotel bookings dataset, examples include:

- `lead_time` cannot be negative.
- `stays_in_weekend_nights` cannot be negative.
- `stays_in_week_nights` cannot be negative.
- `total_of_special_requests` cannot be negative.

In [8]:
business_rule_valid = (
    (df['lead_time'] >= 0) &
    (df['stays_in_weekend_nights'] >= 0) &
    (df['stays_in_week_nights'] >= 0) &
    (df['total_of_special_requests'] >= 0)
)

print("Records violating business rules:", (~business_rule_valid).sum())

Records violating business rules: 0


### Observation

Business rules were applied to important booking-related fields. Records that violate the defined business rules were identified for further review.

## 8. Referential Validation

Referential validation checks whether values in a column correctly refer to valid values or predefined references.

For the hotel bookings dataset, columns such as `hotel`, `meal`, and `customer_type` can be checked against their accepted values to ensure that the data contains valid references.

In [9]:
valid_meals = ['BB', 'FB', 'HB', 'SC', 'Undefined']

valid_customer_types = [
    'Contract',
    'Group',
    'Transient',
    'Transient-Party'
]

meal_valid = df['meal'].isin(valid_meals)
customer_type_valid = df['customer_type'].isin(valid_customer_types)

print("Invalid meal references:", (~meal_valid).sum())
print("Invalid customer type references:", (~customer_type_valid).sum())

Invalid meal references: 0
Invalid customer type references: 0


### Observation

Referential validation was applied to categorical reference values. Any values that do not match the predefined valid references were identified as invalid.

## 9. Date Validation

Date validation checks whether date values are valid and follow the expected date rules.

For the hotel bookings dataset, `reservation_status_date` should contain valid dates that can be correctly converted into a date format.

In [10]:
date_values = pd.to_datetime(
    df['reservation_status_date'],
    errors='coerce'
)

valid_dates = date_values.notna()

print("Valid dates:", valid_dates.sum())
print("Invalid dates:", (~valid_dates).sum())

Valid dates: 119390
Invalid dates: 0


### Observation

The `reservation_status_date` column was checked to identify valid and invalid date values. Invalid dates were identified using date conversion with error handling.

## 10. Constraint Validation

Constraint validation checks whether multiple data conditions are satisfied at the same time.

For the hotel bookings dataset, constraints can be applied to important booking fields such as `lead_time`, `adults`, `children`, and `adr`.

A record is considered valid only when it satisfies all the defined constraints.

In [11]:
constraints_valid = (
    (df['lead_time'] >= 0) &
    (df['adults'] >= 0) &
    (df['children'] >= 0) &
    (df['adr'] >= 0)
)

print("Valid records:", constraints_valid.sum())
print("Invalid records:", (~constraints_valid).sum())

Valid records: 119385
Invalid records: 5


### Observation

Multiple constraints were combined to validate the hotel booking records. Records that failed one or more constraints were identified as invalid.